In [10]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/luisfelipevendramim/m5-processed/m5_processed.csv


**1. Importações**

In [11]:
from sklearn.preprocessing import LabelEncoder

import warnings
warnings.filterwarnings("ignore")

**2. Carregar Dados**

In [12]:
df = pd.read_csv(
    "/kaggle/input/datasets/luisfelipevendramim/m5-processed/m5_processed.csv",
    parse_dates=["date"]
)

df.head()

,store_id,cat_id,d,sales,date,wm_yr_wk,weekday,month,year,event_name_1,event_type_1,snap_CA,snap_TX,snap_WI
0,CA_1,FOODS,d_1,3239,2011-01-29,11101,Saturday,1,2011,NaN,NaN,0,0,0
1,CA_1,HOBBIES,d_1,556,2011-01-29,11101,Saturday,1,2011,NaN,NaN,0,0,0
2,CA_1,HOUSEHOLD,d_1,542,2011-01-29,11101,Saturday,1,2011,NaN,NaN,0,0,0
3,CA_2,FOODS,d_1,2193,2011-01-29,11101,Saturday,1,2011,NaN,NaN,0,0,0
4,CA_2,HOBBIES,d_1,538,2011-01-29,11101,Saturday,1,2011,NaN,NaN,0,0,0


**3. Ordenação**

In [13]:
df = df.sort_values(
    ["store_id", "cat_id", "date"]
).reset_index(drop=True)

**4. Criar Chave da Série** (Loja + Categoria)

In [15]:
df["series_id"] = (
    df["store_id"] + "_" + df["cat_id"]
)

**5. Features de Calendário**

In [16]:
df["day"] = df["date"].dt.day

df["week"] = df["date"].dt.isocalendar().week.astype(int)

df["quarter"] = df["date"].dt.quarter

df["dayofweek"] = df["date"].dt.dayofweek

df["is_weekend"] = (
    df["dayofweek"] >= 5
).astype(int)

**6. Features de Lag**

In [19]:
df["lag_7"] = (
    df.groupby("series_id")["sales"]
    .shift(7)
)

df["lag_14"] = (
    df.groupby("series_id")["sales"]
    .shift(14)
)

df["lag_28"] = (
    df.groupby("series_id")["sales"]
    .shift(28)
)

df["lag_56"] = (
    df.groupby("series_id")["sales"]
    .shift(56)
)

**7. Rolling Mean**

In [20]:
df["rolling_mean_7"] = (
    df.groupby("series_id")["sales"]
    .transform(
        lambda x:
        x.shift(1)
        .rolling(7)
        .mean()
    )
)

df["rolling_mean_28"] = (
    df.groupby("series_id")["sales"]
    .transform(
        lambda x:
        x.shift(1)
        .rolling(28)
        .mean()
    )
)

**8. Rolling Std**

In [21]:
df["rolling_std_7"] = (
    df.groupby("series_id")["sales"]
    .transform(
        lambda x:
        x.shift(1)
        .rolling(7)
        .std()
    )
)

df["rolling_std_28"] = (
    df.groupby("series_id")["sales"]
    .transform(
        lambda x:
        x.shift(1)
        .rolling(28)
        .std()
    )
)

**9. Tendência de Crescimento**

In [22]:
df["trend_7"] = (
    df["rolling_mean_7"]
    - df["rolling_mean_28"]
)

**10. Encoding de Variáveis Categóricas**

In [23]:
le_store = LabelEncoder()

df["store_encoded"] = (
    le_store.fit_transform(
        df["store_id"]
    )
)

le_cat = LabelEncoder()

df["cat_encoded"] = (
    le_cat.fit_transform(
        df["cat_id"]
    )
)

df["event_type_1"] = (
    df["event_type_1"]
    .fillna("None")
)

le_event = LabelEncoder()

df["event_encoded"] = (
    le_event.fit_transform(
        df["event_type_1"]
    )
)

**11. Features de Sazonalidade Cíclica** (Transformando mês em seno | cosseno)

In [24]:
df["month_sin"] = np.sin(
    2*np.pi*df["month"]/12
)

df["month_cos"] = np.cos(
    2*np.pi*df["month"]/12
)

Dia da semana

In [25]:
df["dow_sin"] = np.sin(
    2*np.pi*df["dayofweek"]/7
)

df["dow_cos"] = np.cos(
    2*np.pi*df["dayofweek"]/7
)

**12. Remover Linhas Incompletas**

In [26]:
df = df.dropna()

df.shape

(4410, 36)

**13. Selecionar Features**

In [27]:
features = [
    "store_encoded",
    "cat_encoded",
    "month",
    "week",
    "quarter",
    "dayofweek",
    "is_weekend",
    "snap_CA",
    "snap_TX",
    "snap_WI",
    "event_encoded",
    "lag_7",
    "lag_14",
    "lag_28",
    "lag_56",
    "rolling_mean_7",
    "rolling_mean_28",
    "rolling_std_7",
    "rolling_std_28",
    "trend_7",
    "month_sin",
    "month_cos",
    "dow_sin",
    "dow_cos"
]

Target:

In [28]:
target = "sales"

**14. Salvar Base Final**

In [29]:
df.to_csv(
    "m5_feature_engineered.csv",
    index=False
)